# Data Cleaning 07 -- OpenAssetPricing Monthly

## Input
`Data/Data_Collection/Initial/07_OpenAssetPricing/firm_monthly/` (partitioned parquet by year, 48,861 rows across 2004--2024, 227 PERMNOs, ~209 factor columns)

## Purpose
Cleans monthly stock-level factor data from OpenAssetPricing -- the largest factor set in the pipeline. NaN analysis is conducted at three levels: per-factor (which factors are too sparse?), per-factor-by-year (are factors missing because they start late or stop early?), and per-PERMNO (are certain stocks systematically missing?). Overlap with IBES-collected factors is also assessed.

## Stage 0: Load & Inspect
- Loads all year partitions and verifies PERMNO coverage against the master list
- Reports shape, date range, unique PERMNOs, and rows per year with approximate stocks-per-month count
- Separates columns into ID, metadata, and factor groups

## Stage 1: Per-Factor NaN Rates
- Computes NaN rate for all ~209 factor columns and classifies into tiers: 0% NaN, <5%, 5--30%, and >=30% (drop candidates)
- Full listing of factors in the >=30% tier and the 5--30% tier with exact NaN counts

## Stage 2: Per-Factor NaN Rates by Year
- For factors in the 5--30% NaN tier, prints a compact heatmap of NaN percentage by year to reveal temporal patterns (late-starting, early-stopping, or uniformly sparse)
- Identifies late-starting factors (first year with <50% NaN is after 2004)
- Also checks the <5% tier for factors that are secretly late-starting (low overall NaN but 100% NaN in early years)

## Stage 3: Per-PERMNO NaN Rates
- Computes average NaN rate per PERMNO using only the retained factors
- Reports distribution across PERMNOs and lists the 20 worst
- Cross-references high-NaN PERMNOs against `universe_annual` to determine whether they are short-lived stocks in the top-100

## Stage 4: Overlap with IBES Factors
- Identifies OAP factors that potentially overlap with the IBES-collected data (keyword matching on analyst/forecast/earnings/recommendation terms)
- Reports whether each overlapping factor is being dropped (>=30% NaN) or kept, to inform the decision of which source to use

## Stage 5: Duplicate & Coverage Checks
- Duplicate `(permno, date)` check
- Date frequency verification (month-end)
- Coverage per PERMNO: months of data, identifies PERMNOs with fewer than 24 months
- Stocks per month distribution

## Stage 6b: In-Universe NaN Rates
- Merges with `universe_annual` to restrict analysis to only observations where the stock was actually in the top-100 that year
- Recomputes per-factor NaN rates for in-universe observations and compares against overall rates
- Confirms no retained factors exceed 30% NaN when restricted to in-universe observations
- Recomputes per-PERMNO NaN rates for in-universe observations only, lists the 15 worst

## Stage 7: Clean & Save

### Columns Dropped (70)
- **57 factors with >=30% NaN** across the full dataset. These are niche academic factors (ProbInformedTrading, Governance, Activism, R&D-derived, etc.) that are not computed for most large-cap S&P 500 stocks.
- **7 options-derived factors** (`skew1`, `CPVolSpread`, `RIVolSpread`, `dVolPut`, `dVolCall`, `dCPVolSpread`, `SmileSlope`) that stop being computed in ~2022, leaving 2023--2024 with >90% NaN. Replaced by the project's own OptionMetrics data from notebook 10.
- **6 IBES-overlapping factors** (`AnalystRevision`, `AnalystValue`, `ForecastDispersion`, `EarningsForecastDisparity`, `EarningsStreak`, `FEPS`) where the project's own IBES collection provides more complete monthly coverage.

### No Winsorisation
Applied cross-sectionally in the merge pipeline.

### No Forward-Fill
Stock-level monthly data -- cross-sectional aggregation skips NaN naturally.

### Structural NaN Left As-Is
Remaining NaN comes from accounting factors waiting for annual filings (OperProf, GP, etc.), short-tenure stocks with limited factor history, and factors requiring long lookback periods (e.g., MomSeason16YrPlus needs 16 years of history).

### In-Universe NaN Tiers (After Dropping)
0% NaN: 1, <5%: 97, 5--15%: 30, 15--30%: 12, >=30%: 0.

## Output
`Data/Data_Collection/Cleaned/07_OpenAssetPricing/oap_monthly_clean.parquet` -- 140 factor columns (down from ~209), 48,861 rows

In [1]:
# %% [markdown]
# # Data Cleaning: OpenAssetPricing Monthly (firm_monthly)
#
# Source: Data/Data_Collection/Initial/07_OpenAssetPricing/firm_monthly/ (partitioned by year)
# Output: Data/Data_Collection/Cleaned/07_OpenAssetPricing/oap_monthly_clean.parquet
#
# Monthly stock-level factor data from OpenAssetPricing. ~209 pre-computed
# academic factors (momentum, value, profitability, etc.) for all PERMNOs
# in the master universe. Already filtered during collection.
#
# This is the largest factor set — needs careful NaN analysis at three levels:
#   1. Per-factor (which factors are too sparse to use?)
#   2. Per-factor-by-year (are factors missing because they start late?)
#   3. Per-PERMNO (are certain stocks systematically missing?)

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR     = Path('../../../Data/Data_Collection/Initial/07_OpenAssetPricing/firm_monthly/')
MASTER_PATH = Path('../../../Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_master_clean.parquet')
ANNUAL_PATH = Path('../../../Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_annual_clean.parquet')
OUT_DIR     = Path('../../../Data/Data_Collection/Cleaned/07_OpenAssetPricing')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD & INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD & INSPECT — OAP Monthly")
print("=" * 90)

df = pd.read_parquet(RAW_DIR)
df['date'] = pd.to_datetime(df['date'])
master = pd.read_parquet(MASTER_PATH)

print(f"\n  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Unique dates: {df['date'].nunique():,}")
print(f"  Unique PERMNOs: {df['permno'].nunique()}")
print(f"  Master PERMNOs: {len(master)}")

# Verify PERMNOs
data_permnos = set(df['permno'].unique())
master_permnos = set(master['permno'])
extra = data_permnos - master_permnos
missing = master_permnos - data_permnos
print(f"\n  PERMNOs in data but NOT in master: {len(extra)}")
print(f"  PERMNOs in master but NOT in data: {len(missing)}")
if missing:
    print(f"    Missing: {sorted(missing)}")

# Identify column types
all_cols = df.columns.tolist()
id_cols = ['permno']
date_cols = ['date']
meta_cols = [c for c in ['year'] if c in all_cols]
factor_cols = [c for c in all_cols if c not in id_cols + date_cols + meta_cols]

print(f"\n  Total factor columns: {len(factor_cols)}")
print(f"  Meta columns: {meta_cols}")

# Rows per year
print(f"\n--- Rows per year ---")
rows_per_year = df.groupby(df['date'].dt.year).size()
for year, n in rows_per_year.items():
    avg_stocks = n / 12
    print(f"  {year}: {n:>6,d} rows (~{avg_stocks:.0f} stocks/month)")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: PER-FACTOR NaN RATES
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: PER-FACTOR NaN RATES")
print("=" * 90)

n_rows = len(df)

# ── Compute NaN rates ────────────────────────────────────────────────────────
col_nan = df[factor_cols].isna().sum()
col_nan_pct = (col_nan / n_rows * 100).round(2)
col_nan_sorted = col_nan_pct.sort_values(ascending=False)

# ── Summary by tier ──────────────────────────────────────────────────────────
tier_0 = col_nan_pct[col_nan_pct == 0]
tier_clean = col_nan_pct[(col_nan_pct > 0) & (col_nan_pct < 5)]
tier_impute = col_nan_pct[(col_nan_pct >= 5) & (col_nan_pct < 30)]
tier_drop = col_nan_pct[col_nan_pct >= 30]

print(f"\n  Factors with   0% NaN: {len(tier_0)}")
print(f"  Factors with  <5% NaN: {len(tier_clean)}")
print(f"  Factors with 5-30% NaN: {len(tier_impute)}")
print(f"  Factors with ≥30% NaN: {len(tier_drop)}  ← DROP")
print(f"  ──────────────────────────")
print(f"  Total factors: {len(factor_cols)}")
print(f"  Factors to KEEP: {len(tier_0) + len(tier_clean) + len(tier_impute)}")
print(f"  Factors to DROP: {len(tier_drop)}")

# ── Full table: factors to DROP ──────────────────────────────────────────────
if len(tier_drop) > 0:
    print(f"\n--- Factors to DROP (≥30% NaN): {len(tier_drop)} ---")
    print(f"\n  {'Factor':<40s} {'NaN %':>8s}  {'Count':>8s}")
    print("  " + "-" * 60)
    for col in tier_drop.sort_values(ascending=False).index:
        pct = col_nan_pct[col]
        count = int(col_nan[col])
        print(f"  {col:<40s} {pct:>7.2f}%  {count:>8,d}")

# ── Full table: factors to KEEP (5-30% NaN) ─────────────────────────────────
if len(tier_impute) > 0:
    print(f"\n--- Factors to KEEP but with notable NaN (5-30%): {len(tier_impute)} ---")
    print(f"\n  {'Factor':<40s} {'NaN %':>8s}  {'Count':>8s}")
    print("  " + "-" * 60)
    for col in tier_impute.sort_values(ascending=False).index:
        pct = col_nan_pct[col]
        count = int(col_nan[col])
        print(f"  {col:<40s} {pct:>7.2f}%  {count:>8,d}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: PER-FACTOR NaN RATES BY YEAR
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 2: PER-FACTOR NaN RATES BY YEAR")
print("=" * 90)

# Focus on the 5-30% tier — these might be late-starting factors
keep_factors = list(tier_impute.sort_values(ascending=False).index)
df['_year'] = df['date'].dt.year
years = sorted(df['_year'].unique())

# ── Heatmap: NaN % by factor × year for 5-30% tier ──────────────────────────
print(f"\n--- NaN % by year for factors in the 5-30% range ---")
print(f"    (showing factors that might start late or have temporal patterns)")

if len(keep_factors) > 0:
    nan_by_year = df.groupby('_year')[keep_factors].apply(
        lambda x: x.isna().mean() * 100
    ).round(1)

    # Print a compact table: factor name + NaN% for each year
    header = f"  {'Factor':<35s}" + "".join(f" {y:>5d}" for y in years)
    print(f"\n{header}")
    print("  " + "-" * (35 + 6 * len(years)))

    for col in keep_factors[:40]:  # show top 40
        row = f"  {col:<35s}"
        for y in years:
            pct = nan_by_year.loc[y, col] if y in nan_by_year.index else 0
            if pct >= 90:
                row += f"  {'--':>5s}"
            elif pct >= 30:
                row += f" {pct:>4.0f}%"
            elif pct > 0:
                row += f" {pct:>4.1f}"
            else:
                row += f"  {'·':>5s}"
        print(row)

# ── Identify late-starting factors ───────────────────────────────────────────
print(f"\n--- Late-starting factors (first year with <50% NaN) ---")
for col in keep_factors:
    yearly_nan = df.groupby('_year')[col].apply(lambda x: x.isna().mean() * 100)
    first_good = yearly_nan[yearly_nan < 50]
    if len(first_good) > 0 and first_good.index[0] > years[0]:
        print(f"  {col:<35s} starts ~{first_good.index[0]} "
              f"(NaN before: {yearly_nan.loc[:first_good.index[0]-1].mean():.0f}%)")

# ── Also check the <5% tier for any that are secretly late-starting ──────────
print(f"\n--- Clean factors (<5% NaN) that are actually late-starting ---")
clean_factors = list(tier_clean.index) + list(tier_0.index)
for col in clean_factors:
    yearly_nan = df.groupby('_year')[col].apply(lambda x: x.isna().mean() * 100)
    first_good = yearly_nan[yearly_nan < 50]
    if len(first_good) > 0 and first_good.index[0] > years[0]:
        overall = col_nan_pct[col]
        print(f"  {col:<35s} starts ~{first_good.index[0]} (overall NaN: {overall:.1f}%)")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: PER-PERMNO NaN RATES
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 3: PER-PERMNO NaN RATES")
print("=" * 90)

# ── Compute per-PERMNO NaN using ONLY the factors we're keeping ──────────────
keep_factor_list = list(tier_0.index) + list(tier_clean.index) + list(tier_impute.index)

print(f"\n  Using {len(keep_factor_list)} retained factors (excluding {len(tier_drop)} dropped)")

permno_nan = (
    df.groupby('permno')[keep_factor_list]
    .apply(lambda x: x.isna().mean().mean() * 100)
    .sort_values(ascending=False)
)

print(f"\n  PERMNOs with <5% avg NaN:    {(permno_nan < 5).sum()}")
print(f"  PERMNOs with 5-10% avg NaN:  {((permno_nan >= 5) & (permno_nan < 10)).sum()}")
print(f"  PERMNOs with 10-20% avg NaN: {((permno_nan >= 10) & (permno_nan < 20)).sum()}")
print(f"  PERMNOs with >20% avg NaN:   {(permno_nan >= 20).sum()}")

# ── Worst PERMNOs ────────────────────────────────────────────────────────────
worst = permno_nan.head(20)
if len(worst) > 0:
    print(f"\n  20 worst PERMNOs (after dropping bad factors):")
    print(f"  {'PERMNO':>8s}  {'Avg NaN %':>10s}  {'Rows':>6s}  {'Years':>6s}")
    print("  " + "-" * 35)
    for permno, pct in worst.items():
        n_rows_p = len(df[df['permno'] == permno])
        first_yr = df[df['permno'] == permno]['_year'].min()
        last_yr = df[df['permno'] == permno]['_year'].max()
        print(f"  {int(permno):>8d}  {pct:>9.2f}%  {n_rows_p:>6,d}  {first_yr}-{last_yr}")

# ── Are worst PERMNOs short-lived stocks? ────────────────────────────────────
annual = pd.read_parquet(ANNUAL_PATH)
print(f"\n--- Are high-NaN PERMNOs in the universe for few years? ---")
for permno in permno_nan[permno_nan > 15].index:
    n_universe_years = len(annual[annual['permno'] == permno])
    n_data_years = df[df['permno'] == permno]['_year'].nunique()
    total_rows = len(df[df['permno'] == permno])
    print(f"  PERMNO {int(permno):>6d}: {n_universe_years:>2d} yrs in top-100, "
          f"{n_data_years:>2d} yrs of data, {total_rows:,} rows, "
          f"{permno_nan[permno]:.1f}% NaN")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: OVERLAP WITH IBES
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 4: OVERLAP WITH IBES FACTORS")
print("=" * 90)

# OAP contains some analyst-derived factors that overlap with our IBES data
# Identify them so we can decide whether to keep OAP's version or use ours
analyst_keywords = ['Recomm', 'Analyst', 'Forecast', 'Earnings', 'Revenue',
                    'FEPS', 'Surprise', 'Consensus', 'ChangeInRecommendation',
                    'UpRecomm', 'DownRecomm', 'ConsRecomm']

print(f"\n--- OAP factors potentially overlapping with IBES ---")
for col in factor_cols:
    for kw in analyst_keywords:
        if kw.lower() in col.lower():
            pct = col_nan_pct.get(col, 0)
            status = "DROPPING (≥30% NaN)" if pct >= 30 else f"KEEPING ({pct:.1f}% NaN)"
            print(f"  {col:<40s} {status}")
            break

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 5: DUPLICATE & COVERAGE CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 5: DUPLICATE & COVERAGE CHECKS")
print("=" * 90)

# ── Duplicate (permno, date) ─────────────────────────────────────────────────
print(f"\n--- Duplicate (permno, date) ---")
n_dupes = df.duplicated(subset=['permno', 'date']).sum()
if n_dupes == 0:
    print(f"  ✓ No duplicates")
else:
    print(f"  ⚠ {n_dupes} duplicates")

# ── Date frequency ───────────────────────────────────────────────────────────
print(f"\n--- Date frequency ---")
is_month_end = df['date'].dt.is_month_end
print(f"  Dates that are month-end: {is_month_end.sum():,} ({is_month_end.mean()*100:.1f}%)")

# ── Coverage per PERMNO ──────────────────────────────────────────────────────
print(f"\n--- Coverage per PERMNO ---")
permno_coverage = df.groupby('permno').agg(
    n_months=('date', 'nunique'),
    first_date=('date', 'min'),
    last_date=('date', 'max')
)
print(f"  Months per PERMNO: mean={permno_coverage['n_months'].mean():.0f}, "
      f"median={permno_coverage['n_months'].median():.0f}, "
      f"min={permno_coverage['n_months'].min()}, "
      f"max={permno_coverage['n_months'].max()}")

short = permno_coverage[permno_coverage['n_months'] < 24]
if len(short) > 0:
    print(f"\n  PERMNOs with <24 months: {len(short)}")
    for permno, row in short.iterrows():
        print(f"    PERMNO {int(permno):>6d}  {row['n_months']:>3d} months "
              f"({row['first_date'].date()} → {row['last_date'].date()})")

# ── Stocks per month ─────────────────────────────────────────────────────────
print(f"\n--- Stocks per month ---")
stocks_per_month = df.groupby('date')['permno'].nunique()
print(f"  Mean: {stocks_per_month.mean():.1f}")
print(f"  Min:  {stocks_per_month.min()} (on {stocks_per_month.idxmin().date()})")
print(f"  Max:  {stocks_per_month.max()} (on {stocks_per_month.idxmax().date()})")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 6: SUMMARY — DECISIONS NEEDED
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 6: SUMMARY")
print("=" * 90)

drop_factors = list(tier_drop.index)
keep_factors_final = keep_factor_list

print(f"""
FACTOR TIERS:
  0% NaN (perfect):     {len(tier_0):>4d} factors
  <5% NaN (clean):      {len(tier_clean):>4d} factors
  5-30% NaN (usable):   {len(tier_impute):>4d} factors
  ≥30% NaN (DROP):      {len(tier_drop):>4d} factors
  ────────────────────────
  TOTAL KEEP:           {len(keep_factors_final):>4d}
  TOTAL DROP:           {len(drop_factors):>4d}

CLEANING PLAN:
  1. Drop {len(drop_factors)} factors with ≥30% NaN
  2. Leave structural NaN in remaining factors as-is
  3. No winsorisation (done in merge pipeline)
  4. No forward-fill (stock-level monthly data)
  5. Cross-sectional aggregation will skip NaN

Drop the ≥30% factors and save. Paste back the output
and I will write the cleaning cell.
""")

# Clean up temp column
df = df.drop(columns='_year', errors='ignore')

STAGE 0: LOAD & INSPECT — OAP Monthly

  Shape: 48,861 rows × 213 columns
  Date range: 2004-01-31 → 2024-12-31
  Unique dates: 252
  Unique PERMNOs: 227
  Master PERMNOs: 227

  PERMNOs in data but NOT in master: 0
  PERMNOs in master but NOT in data: 0

  Total factor columns: 210
  Meta columns: ['year']

--- Rows per year ---
  2004:  2,390 rows (~199 stocks/month)
  2005:  2,398 rows (~200 stocks/month)
  2006:  2,377 rows (~198 stocks/month)
  2007:  2,393 rows (~199 stocks/month)
  2008:  2,402 rows (~200 stocks/month)
  2009:  2,379 rows (~198 stocks/month)
  2010:  2,375 rows (~198 stocks/month)
  2011:  2,351 rows (~196 stocks/month)
  2012:  2,361 rows (~197 stocks/month)
  2013:  2,393 rows (~199 stocks/month)
  2014:  2,386 rows (~199 stocks/month)
  2015:  2,385 rows (~199 stocks/month)
  2016:  2,361 rows (~197 stocks/month)
  2017:  2,340 rows (~195 stocks/month)
  2018:  2,317 rows (~193 stocks/month)
  2019:  2,280 rows (~190 stocks/month)
  2020:  2,225 rows (~185 st

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 6b: IN-UNIVERSE NaN RATES
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 6b: IN-UNIVERSE NaN RATES")
print("=" * 90)

annual = pd.read_parquet(ANNUAL_PATH)

# ── Define the factors we're actually keeping ────────────────────────────────
# Drop: 57 (≥30% NaN) + 7 (options stopping early) + IBES overlaps
drop_30pct = list(tier_drop.index)

drop_options_stopping = ['skew1', 'CPVolSpread', 'RIVolSpread', 'dVolPut',
                         'dVolCall', 'dCPVolSpread', 'SmileSlope']

drop_ibes_overlap = ['AnalystRevision', 'AnalystValue', 'ForecastDispersion',
                     'EarningsForecastDisparity', 'EarningsStreak', 'FEPS']

all_drops = set(drop_30pct + drop_options_stopping + drop_ibes_overlap)
all_drops_present = [c for c in all_drops if c in df.columns]
keep_factors = [c for c in factor_cols if c not in all_drops]

print(f"\n  Factors dropped (≥30% NaN):       {len(drop_30pct)}")
print(f"  Factors dropped (options stop):   {len([c for c in drop_options_stopping if c in df.columns])}")
print(f"  Factors dropped (IBES overlap):   {len([c for c in drop_ibes_overlap if c in df.columns])}")
print(f"  Total dropped:                    {len(all_drops_present)}")
print(f"  Total kept:                       {len(keep_factors)}")

# ── Filter to in-universe observations only ──────────────────────────────────
df['_year'] = df['date'].dt.year
df_universe = df.merge(
    annual[['permno', 'year']],
    left_on=['permno', '_year'],
    right_on=['permno', 'year'],
    how='inner'
)
print(f"\n  All rows: {len(df):,}")
print(f"  In-universe rows: {len(df_universe):,}")
print(f"  Dropped (out of universe): {len(df) - len(df_universe):,}")

# ── Per-factor NaN rates: in-universe only ───────────────────────────────────
n_univ = len(df_universe)
univ_nan = df_universe[keep_factors].isna().sum()
univ_nan_pct = (univ_nan / n_univ * 100).round(2)
univ_nan_sorted = univ_nan_pct.sort_values(ascending=False)

# Compare with overall rates
overall_nan_pct = (df[keep_factors].isna().sum() / len(df) * 100).round(2)

print(f"\n--- Per-Factor NaN: In-Universe vs Overall (kept factors only) ---")
print(f"\n  {'Factor':<35s} {'Universe':>10s}  {'Overall':>10s}  {'Diff':>8s}")
print("  " + "-" * 68)
for col in univ_nan_sorted.index:
    u_pct = univ_nan_pct[col]
    o_pct = overall_nan_pct[col]
    diff = u_pct - o_pct
    if u_pct > 0:
        print(f"  {col:<35s} {u_pct:>9.2f}%  {o_pct:>9.2f}%  {diff:>+7.2f}%")

# ── Summary tiers for in-universe ────────────────────────────────────────────
u_tier_0 = univ_nan_pct[univ_nan_pct == 0]
u_tier_clean = univ_nan_pct[(univ_nan_pct > 0) & (univ_nan_pct < 5)]
u_tier_mid = univ_nan_pct[(univ_nan_pct >= 5) & (univ_nan_pct < 15)]
u_tier_high = univ_nan_pct[(univ_nan_pct >= 15) & (univ_nan_pct < 30)]
u_tier_drop = univ_nan_pct[univ_nan_pct >= 30]

print(f"\n--- In-Universe NaN Tiers (kept factors) ---")
print(f"  0% NaN:       {len(u_tier_0)}")
print(f"  <5% NaN:      {len(u_tier_clean)}")
print(f"  5-15% NaN:    {len(u_tier_mid)}")
print(f"  15-30% NaN:   {len(u_tier_high)}")
print(f"  ≥30% NaN:     {len(u_tier_drop)}  ← should be zero!")

if len(u_tier_drop) > 0:
    print(f"\n  ⚠ Factors that are ≥30% NaN even in-universe:")
    for col in u_tier_drop.index:
        print(f"    {col:<35s} {u_tier_drop[col]:.2f}%")

# ── Per-PERMNO NaN: in-universe only ────────────────────────────────────────
print(f"\n--- Per-PERMNO NaN (in-universe, kept factors) ---")
permno_nan_univ = (
    df_universe.groupby('permno')[keep_factors]
    .apply(lambda x: x.isna().mean().mean() * 100)
    .sort_values(ascending=False)
)

print(f"  PERMNOs with <5% avg NaN:    {(permno_nan_univ < 5).sum()}")
print(f"  PERMNOs with 5-10% avg NaN:  {((permno_nan_univ >= 5) & (permno_nan_univ < 10)).sum()}")
print(f"  PERMNOs with 10-20% avg NaN: {((permno_nan_univ >= 10) & (permno_nan_univ < 20)).sum()}")
print(f"  PERMNOs with >20% avg NaN:   {(permno_nan_univ >= 20).sum()}")

worst = permno_nan_univ.head(15)
print(f"\n  15 worst PERMNOs (in-universe only):")
print(f"  {'PERMNO':>8s}  {'Avg NaN %':>10s}  {'Univ Rows':>10s}  {'Yrs in Top100':>14s}")
print("  " + "-" * 48)
for permno, pct in worst.items():
    n_rows_u = len(df_universe[df_universe['permno'] == permno])
    n_yrs = len(annual[annual['permno'] == permno])
    print(f"  {int(permno):>8d}  {pct:>9.2f}%  {n_rows_u:>10,d}  {n_yrs:>14d}")

# Clean up
df = df.drop(columns=['_year'], errors='ignore')

STAGE 6b: IN-UNIVERSE NaN RATES

  Factors dropped (≥30% NaN):       57
  Factors dropped (options stop):   7
  Factors dropped (IBES overlap):   6
  Total dropped:                    70
  Total kept:                       140

  All rows: 48,861
  In-universe rows: 25,194
  Dropped (out of universe): 23,667

--- Per-Factor NaN: In-Universe vs Overall (kept factors only) ---

  Factor                                Universe     Overall      Diff
  --------------------------------------------------------------------
  PredictedFE                             26.20%      29.33%    -3.13%
  OperProf                                25.63%      28.19%    -2.56%
  CBOperProf                              24.39%      29.39%    -5.00%
  NetPayoutYield                          23.78%      28.52%    -4.74%
  AOP                                     22.08%      22.33%    -0.25%
  GP                                      18.77%      22.68%    -3.91%
  GrSaleToGrOverhead                      18.71%     

In [3]:
# %% [markdown]
# ## Stage 7: Clean & Save
#
# **Data overview:**
# Monthly stock-level factor data from OpenAssetPricing. 48,861 rows across
# 2004–2024 for 227 PERMNOs. Already filtered to universe during collection.
#
# **Columns dropped (70):**
# - 57 factors with ≥30% NaN across the full dataset. These are niche academic
#   factors (ProbInformedTrading, Governance, Activism, R&D-derived, etc.) that
#   are not computed for most large-cap S&P 500 stocks.
# - 7 options-derived factors (skew1, CPVolSpread, RIVolSpread, dVolPut,
#   dVolCall, dCPVolSpread, SmileSlope) that stop being computed in ~2022,
#   leaving 2023–2024 with >90% NaN. Replaced by our own OptionMetrics data.
# - 6 IBES-overlapping factors (AnalystRevision, AnalystValue,
#   ForecastDispersion, EarningsForecastDisparity, EarningsStreak, FEPS)
#   where we have more complete monthly data from our own IBES collection.
#
# **Factors retained: 140.** In-universe NaN tiers:
#   0% NaN: 1, <5%: 97, 5–15%: 30, 15–30%: 12, ≥30%: 0.
#
# **No winsorisation.** Applied cross-sectionally in merge pipeline.
#
# **No forward-fill.** Stock-level monthly data — cross-sectional aggregation
# skips NaN naturally.
#
# **Structural NaN left as-is.** Remaining NaN comes from:
# - Accounting factors waiting for annual filings (OperProf, GP, etc.)
# - Short-tenure stocks with limited factor history
# - Factors requiring lookback periods (MomSeason16YrPlus needs 16 years)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 7: CLEAN & SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 7: CLEAN & SAVE")
print("=" * 90)

# ── 7a. Drop columns ────────────────────────────────────────────────────────
drop_30pct = list(tier_drop.index)

drop_options_stopping = ['skew1', 'CPVolSpread', 'RIVolSpread', 'dVolPut',
                         'dVolCall', 'dCPVolSpread', 'SmileSlope']

drop_ibes_overlap = ['AnalystRevision', 'AnalystValue', 'ForecastDispersion',
                     'EarningsForecastDisparity', 'EarningsStreak', 'FEPS']

all_drops = set(drop_30pct + drop_options_stopping + drop_ibes_overlap)
all_drops_present = [c for c in all_drops if c in df.columns]

df = df.drop(columns=all_drops_present)

meta_cols = [c for c in ['year'] if c in df.columns]
factor_cols_final = [c for c in df.columns if c not in ['date', 'permno'] + meta_cols]

print(f"\n  Dropped {len(all_drops_present)} columns:")
print(f"    ≥30% NaN:        {len([c for c in drop_30pct if c in all_drops_present])}")
print(f"    Options stop:    {len([c for c in drop_options_stopping if c in all_drops_present])}")
print(f"    IBES overlap:    {len([c for c in drop_ibes_overlap if c in all_drops_present])}")
print(f"  Remaining factor columns: {len(factor_cols_final)}")

# ── 7b. Final NaN report ────────────────────────────────────────────────────
nan_check = df[factor_cols_final].isna().sum()
nan_cols = nan_check[nan_check > 0].sort_values(ascending=False)
total_nan = nan_cols.sum()
total_cells = len(df) * len(factor_cols_final)
print(f"\n  Total NaN: {total_nan:,} / {total_cells:,} ({total_nan/total_cells*100:.2f}%)")
print(f"  Factors with any NaN: {len(nan_cols)} / {len(factor_cols_final)}")

# ── 7c. Final summary ───────────────────────────────────────────────────────
print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  PERMNOs: {df['permno'].nunique()}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

print(f"\n  Sample (first 5 rows, first 8 factors):")
show_cols = ['permno', 'date'] + factor_cols_final[:8]
print(df[show_cols].head(5).to_string(index=False))

# ── 7d. Save ─────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'oap_monthly_clean.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns "
      f"({len(factor_cols_final)} factors)")

print("\nCleaning complete.")

STAGE 7: CLEAN & SAVE

  Dropped 70 columns:
    ≥30% NaN:        57
    Options stop:    7
    IBES overlap:    6
  Remaining factor columns: 140

  Total NaN: 482,922 / 6,840,540 (7.06%)
  Factors with any NaN: 139 / 140

  Final shape: 48,861 rows × 143 columns
  PERMNOs: 227
  Date range: 2004-01-31 → 2024-12-31

  Sample (first 5 rows, first 8 factors):
 permno       date  yyyymm       AM  AOP  AbnormalAccruals  Accruals  AnnouncementReturn  AssetGrowth    BMdec
  10104 2004-01-31  200401 0.152703  NaN         -0.044813  0.068697            0.012650    -0.024444 0.114328
  10104 2004-02-29  200402 0.165481  NaN         -0.044813  0.068697            0.012650    -0.024444 0.114328
  10104 2004-03-31  200403 0.177479  NaN         -0.044813  0.068697           -0.001925    -0.024444 0.114328
  10104 2004-04-30  200404 0.189311  NaN         -0.044813  0.068697           -0.001925    -0.024444 0.114328
  10104 2004-05-31  200405 0.187686  NaN         -0.044813  0.068697           -0.00